# Clase 031 — Series de tiempo

**Parte 0** · VanderPlas cap. 3 § 3.12.

> 🎯 Parsear, indexar, resamplear y rolling sobre datos temporales.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
fechas = pd.date_range('2024-01-01', '2025-12-31', freq='D')
ventas = pd.Series(
    rng.normal(1000, 200, len(fechas)).cumsum().clip(min=0).astype(int),
    index=fechas,
    name='ventas',
)
print(ventas.head())

## 1️⃣ `pd.to_datetime` — parseo robusto

```python
pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
```

`errors='coerce'` convierte lo no parseable a `NaT` (Not a Time) en vez de lanzar excepción.

In [ ]:
raw = pd.Series(['2024-01-15', '15/02/2024', '2024-03-20', 'foo', '2024-04-10'])
fechas_parsed = pd.to_datetime(raw, format='mixed', errors='coerce')
print(fechas_parsed)
print(f'\nNaT count: {fechas_parsed.isna().sum()}')

## 2️⃣ DatetimeIndex y slicing

Con índice datetime, puedes slicear con strings:

In [ ]:
# Trimestre Q1 2024
q1 = ventas.loc['2024-01':'2024-03']
print(f'Q1 2024: {len(q1)} días')

# Solo enero 2025
ene_25 = ventas.loc['2025-01']
print(f'Enero 2025: {len(ene_25)} días')

# Componentes
print(f'\naño/mes/día/dow de la primera fecha:')
print(f'  year={ventas.index[0].year}')
print(f'  month={ventas.index[0].month}')
print(f'  day_name={ventas.index[0].day_name()}')

## 3️⃣ Resampling — cambiar frecuencia

| Alias | Frecuencia |
|---|---|
| `'D'` | día |
| `'W'` | semana |
| `'ME'` | mes (end) |
| `'QE'` | trimestre |
| `'YE'` | año |
| `'h'` | hora |
| `'min'` | minuto |

Resampling **siempre requiere agregación**: sum, mean, last, ohlc.

In [ ]:
# Diaria → mensual
mensual = ventas.resample('ME').agg(['sum', 'mean', 'std']).round(0)
print(mensual.head())

# Diaria → semanal (suma)
semanal = ventas.resample('W').sum()
print(f'\nsemanas: {len(semanal)}')

## 4️⃣ Rolling windows — medias móviles

Ventana móvil: a cada punto, aplicar función a los últimos N puntos. Suaviza tendencias.

In [ ]:
rolling_7  = ventas.rolling(7).mean()
rolling_30 = ventas.rolling(30).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ventas.plot(ax=ax, alpha=0.4, label='diaria', linewidth=0.7)
rolling_7.plot(ax=ax, label='rolling 7d', linewidth=1.2)
rolling_30.plot(ax=ax, label='rolling 30d', linewidth=1.5)
ax.set_title('Ventas — original vs ventanas móviles')
ax.legend()
ax.set_ylabel('ventas')
plt.tight_layout()
plt.show()

## 5️⃣ `shift` y `diff` — lag y variación

```python
s.shift(1)        # adelanta 1 paso (NaN al inicio)
s.diff(1)         # s - s.shift(1) → cambio absoluto
s.pct_change()    # cambio relativo (%)
```

In [ ]:
df = pd.DataFrame({
    'ventas'      : ventas,
    'lag_1'       : ventas.shift(1),
    'diff_1'      : ventas.diff(1),
    'pct_change'  : ventas.pct_change() * 100,
})
print(df.head(6).round(2))

## 6️⃣ Timezones — `tz_localize` y `tz_convert`

```python
s.tz_localize('UTC')           # asigna TZ (no convierte)
s.tz_convert('America/Santiago')  # convierte a otra TZ
```

Regla: primero **localize** (asigna), luego **convert** (mueve).

In [ ]:
naive = pd.Series([1, 2, 3], index=pd.date_range('2024-01-01', periods=3, freq='h'))
utc = naive.tz_localize('UTC')
scl = utc.tz_convert('America/Santiago')   # -3h o -4h según DST
print('UTC:'); print(utc)
print('Santiago:'); print(scl)

## ✅ Checklist

- [ ] Parseo fechas con `to_datetime(errors='coerce')`
- [ ] Indexo por fecha y sliceo con strings
- [ ] Resampleo a la frecuencia objetivo + agg
- [ ] Uso rolling para suavizar tendencias
- [ ] Sé hacer lag features con `shift`/`diff`

## 📝 Homework

Ver `README.md`. Parseo, slice por trimestre, resample mensual, rolling 7/30 con plot, diff.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.12
- [pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)

➡️ **Siguiente:** [032 — eval y query](../032-pandas-eval-y-query/README.md)